# 04 — Dual-view 1D CNN

**Architecture** (Shallue & Vanderburg 2018 — used by Google's AstroNet):
- Two inputs: `global_view` (2001,) and `local_view` (201,).
- Two parallel conv towers (5 blocks for global, 2 for local).
- Concatenate → 4× FC(512) with dropout → sigmoid.

This applies the **Functional API** + **Wide & Deep** patterns from your DATA 305 Week 5 notes — multiple inputs feeding parallel branches that merge.

Use the script for the canonical training run (it logs to MLflow). This notebook is for poking at the model interactively.

In [ ]:
import tensorflow as tf
from omegaconf import OmegaConf

from exoplanet_hunter.models import build_cnn_dualview

cfg = OmegaConf.load("../conf/model/cnn_dualview.yaml")
model = build_cnn_dualview(cfg, global_input_length=2001, local_input_length=201)
model.summary()

In [ ]:
# Visualise the architecture (requires graphviz)
tf.keras.utils.plot_model(model, show_shapes=True, dpi=70)

In [ ]:
# Load processed views and train a tiny model to verify the pipeline works.
from exoplanet_hunter.training.data_module import (
    LightcurveDataset,
    load_views,
    train_val_test_split,
)

views = load_views("../data/processed/views.npz")
train_v, val_v, test_v = train_val_test_split(views, seed=42)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
)
history = model.fit(
    LightcurveDataset(train_v, batch_size=32, augment=True).to_tf_dataset(),
    validation_data=LightcurveDataset(val_v, batch_size=32).to_tf_dataset(),
    epochs=5,
)

In [ ]:
# MC Dropout uncertainty for one test target
import numpy as np

from exoplanet_hunter.models.uncertainty import mc_dropout_predict

g = test_v.global_views[:1, :, None].astype(np.float32)
l = test_v.local_views[:1,  :, None].astype(np.float32)
result = mc_dropout_predict(model, {"global_view": g, "local_view": l}, n_samples=50)
print(f"prob = {result.mean[0]:.3f} ± {result.std[0]:.3f}")